In [0]:
# ============================================================
# 02_schema_and_dead_letter.py
# Phase 2 — Schema Enforcement + Dead Letter Pattern
# ============================================================

import sys
sys.path.append("/Workspace/Users/jaswanth.vudduru@gmail.com/taxi-medallion-pipeline")
from config import RAW_DATA_PATH

# -----------------------------------------------------------
# STEP 1: Create messy test data to simulate real ingestion
# -----------------------------------------------------------
# Write a mixed JSON file with good rows, null fields, corrupt rows
messy_data = """{"vendor_id":"1","pickup_datetime":"2024-01-01 10:00:00","passenger_count":2,"trip_distance":5.2,"fare_amount":18.5,"payment_type":"1"}
{"vendor_id":"2","pickup_datetime":"2024-01-01 10:15:00","passenger_count":1,"trip_distance":3.1,"fare_amount":12.0,"payment_type":"2"}
{"vendor_id":null,"pickup_datetime":"2024-01-01 10:30:00","passenger_count":1,"trip_distance":2.5,"fare_amount":9.5,"payment_type":"1"}
{"vendor_id":"1","pickup_datetime":null,"passenger_count":3,"trip_distance":7.8,"fare_amount":25.0,"payment_type":"1"}
{this is a completely broken json row:::}
{"vendor_id":"2","pickup_datetime":"2024-01-01 11:00:00","passenger_count":6,"trip_distance":12.0,"fare_amount":45.0,"payment_type":"3"}
{"vendor_id":"1","pickup_datetime":"2024-01-01 11:15:00","passenger_count":2,"trip_distance":4.0,"fare_amount":-5.0,"payment_type":"1"}"""

# Create volume if it doesn't exist
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.taxi_data")

# Write to Unity Catalog Volume (writable on serverless compute)
volume_path = f"{RAW_DATA_PATH}/batch1.json"
dbutils.fs.put(volume_path, messy_data, overwrite=True)
print(f"✅ Messy test data written to {volume_path}")

In [0]:
# In 02_schema_and_dead_letter.py — continued

from utils.bronze_ingestion import run_bronze
from config import RAW_DATA_PATH, BRONZE_PATH, DEAD_LETTER_PATH

run_bronze(spark, f"{RAW_DATA_PATH}/", BRONZE_PATH, DEAD_LETTER_PATH)

# -----------------------------------------------------------
# VERIFY: Show what landed in bronze
# -----------------------------------------------------------
print("\n--- BRONZE TABLE ---")
display(spark.read.format("delta").load(BRONZE_PATH))

# -----------------------------------------------------------
# VERIFY: Show what landed in dead letter
# -----------------------------------------------------------
print("\n--- DEAD LETTER (corrupt rows) ---")
# Read dead letter as text to avoid _corrupt_record issues on serverless
display(spark.read.text(DEAD_LETTER_PATH))
# Should show the broken JSON row: {this is a completely broken json row:::}